# eRayz COD KILLER — sunxds_0.7.8 + COD Complete Dataset
---
Nimmt sunxds_0.7.8 (30.000 FPS Bilder) als Basis
und fine-tuned es auf 4.600 echte Call of Duty Bilder.

**ERGEBNIS:** Ein Modell das CoD-Spieler PERFEKT erkennt.

**ANLEITUNG:**
1. Laufzeit > Laufzeittyp > GPU (T4)
2. Alle Zellen ausfuehren (Shift+Enter)
3. sunxds_0.7.8.pt hochladen wenn gefragt
4. ~25-30 Min warten
5. erayz_cod_v1.onnx wird heruntergeladen

In [ ]:
# SCHRITT 1: Installieren
!pip install -q ultralytics roboflow
print('OK: Ultralytics + Roboflow installiert')

In [ ]:
# SCHRITT 2: GPU Check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('PERFEKT!')
else:
    print('FEHLER: Keine GPU! Gehe zu Laufzeit > Laufzeittyp > GPU (T4)')

In [ ]:
# SCHRITT 3: sunxds_0.7.8.pt hochladen
from google.colab import files
import os

if not os.path.exists('sunxds_0.7.8.pt'):
    print('Lade jetzt sunxds_0.7.8.pt hoch...')
    uploaded = files.upload()
    print(f'Hochgeladen: {list(uploaded.keys())}')
else:
    print('sunxds_0.7.8.pt bereits vorhanden!')

# Verify
size = os.path.getsize('sunxds_0.7.8.pt') / (1024*1024)
print(f'Modell: sunxds_0.7.8.pt ({size:.1f} MB)')

In [ ]:
# SCHRITT 4: COD Complete Dataset herunterladen (4.643 Bilder)
# Dieses Dataset hat echte Call of Duty Spieler-Annotationen!
from roboflow import Roboflow

# Roboflow Public API Key (fuer oeffentliche Datasets)
rf = Roboflow(api_key='your_roboflow_api_key')  # <-- DEINEN API KEY HIER
project = rf.workspace('aimbots-gm16b').project('cod-complet-ubmf3')

# Versuche verschiedene Versionen
try:
    dataset = project.version(1).download('yolov8')
except:
    try:
        dataset = project.version(2).download('yolov8')
    except:
        dataset = project.version(3).download('yolov8')

print(f'Dataset heruntergeladen: {dataset.location}')

# Zeige Infos
import glob
train_imgs = glob.glob(f'{dataset.location}/train/images/*')
val_imgs = glob.glob(f'{dataset.location}/valid/images/*')
print(f'Training: {len(train_imgs)} Bilder')
print(f'Validation: {len(val_imgs)} Bilder')
print(f'TOTAL: {len(train_imgs) + len(val_imgs)} Bilder')

In [ ]:
# SCHRITT 5: data.yaml anpassen
# Das COD Dataset hat evtl. andere Klassennamen
# Wir muessen sicherstellen dass 'player' die Hauptklasse ist
import yaml

yaml_path = f'{dataset.location}/data.yaml'
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print('Original Klassen:', data_cfg.get('names', {}))
print('Anzahl Klassen:', data_cfg.get('nc', '?'))

# Zeige was wir haben
print(f'\nDataset Config:')
print(f'  Train: {data_cfg.get("train", "?")}')
print(f'  Val: {data_cfg.get("val", "?")}')
print(f'  Classes: {data_cfg.get("names", {})}')

In [ ]:
# SCHRITT 6: FINE-TUNING starten!
# sunxds_0.7.8 als Basis → trainiert auf COD Complete
from ultralytics import YOLO

model = YOLO('sunxds_0.7.8.pt')
print(f'Basis-Modell: sunxds_0.7.8')
print(f'Original Klassen: {model.names}')
print(f'\nStarte Fine-Tuning auf COD Dataset...')
print(f'Das dauert ca. 25-30 Minuten.\n')

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    name='erayz_cod_v1',
    # Fine-Tuning: Niedrige Lernrate (Basis beibehalten)
    lr0=0.0008,
    lrf=0.01,
    warmup_epochs=3,
    # Erste 10 Layer einfrieren (generelles FPS-Wissen behalten)
    freeze=10,
    # Augmentation (konservativ fuer Fine-Tuning)
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.7,
    mixup=0.05,
    translate=0.1,
    scale=0.3,
)
print('\nTRAINING FERTIG!')

In [ ]:
# SCHRITT 7: Ergebnisse
from IPython.display import Image, display
import os

train_dir = 'runs/detect/erayz_cod_v1'
if not os.path.exists(train_dir):
    for d in sorted(os.listdir('runs/detect')):
        if 'erayz' in d or 'cod' in d:
            train_dir = f'runs/detect/{d}'

print(f'Ergebnisse in: {train_dir}')

# Metriken
if os.path.exists(f'{train_dir}/results.png'):
    display(Image(filename=f'{train_dir}/results.png', width=800))

# Predictions auf Validierung
if os.path.exists(f'{train_dir}/val_batch0_pred.jpg'):
    print('\nValidation Predictions:')
    display(Image(filename=f'{train_dir}/val_batch0_pred.jpg', width=800))

# Confusion Matrix
if os.path.exists(f'{train_dir}/confusion_matrix.png'):
    print('\nConfusion Matrix:')
    display(Image(filename=f'{train_dir}/confusion_matrix.png', width=500))

In [ ]:
# SCHRITT 8: ONNX Export
import shutil

best_model = YOLO(f'{train_dir}/weights/best.pt')

best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
)

onnx_src = f'{train_dir}/weights/best.onnx'
final_name = 'erayz_cod_v1.onnx'
shutil.copy(onnx_src, final_name)

# Auch .pt speichern (fuer spaeteres Weitertraining)
shutil.copy(f'{train_dir}/weights/best.pt', 'erayz_cod_v1.pt')

size_mb = os.path.getsize(final_name) / (1024*1024)
print(f'\n{"="*50}')
print(f'FERTIG: {final_name} ({size_mb:.1f} MB)')
print(f'{"="*50}')
print(f'\nDieses Modell kombiniert:')
print(f'  - sunxds_0.7.8 (30.000 FPS Bilder)')
print(f'  - COD Complete (4.643 Call of Duty Bilder)')
print(f'\nLege erayz_cod_v1.onnx in Downloads/backend/')
print(f'Der Aimbot findet es automatisch!')

In [ ]:
# SCHRITT 9: Download
from google.colab import files
files.download('erayz_cod_v1.onnx')
files.download('erayz_cod_v1.pt')
print('Download gestartet! Beide Dateien speichern.')
print('  - .onnx = fuer den Aimbot')
print('  - .pt = fuer spaeteres Weitertraining')